In [ ]:
%reload_ext autoreload
%autoreload 3

import os
import glob

from lumachords.notation_placer import NotationPlacer
from lumachords.midi_tracker import MidiTracker
from lumachords.runtime_config import AppMode, LogLevel, ProdMode, RuntimeConfig
from lumachords.image_input import ImagePreprocessor
from lumachords.preferences import Preferences
from lumachords.processor import Processor
from lumachords.image_utils import ImageUtils
from lumachords.utils import Utils

actual_fps = 10
apply_lag_to_edge = True
play_y_lag_time_delta = 0.0
active_note_placer = NotationPlacer(crop_silence_at_start=False, min_measures=1, default_split_midi_num=60, always_use_latest_hands_range=True, include_image_alpha_channel=False)

async def detect_keybed(processor: Processor, file_list):
    last_keybed_output = None
    last_image_input = None
    for im_file in file_list:
        kb_im_bgr = ImageUtils.read_image(im_file)
        kb_image_input = await ImagePreprocessor.preprocess_for_keybed(kb_im_bgr)
        keybed_output = await processor.detect_keys(kb_image_input)
        if keybed_output is not None and keybed_output.evaluation_result is None:
            return keybed_output
        last_keybed_output = keybed_output
        last_image_input = last_image_input
    if not keybed_output.evaluation_result and last_image_input is not None:
        if processor.keybed_detection_log_level > LogLevel.LOGLEVEL_INFO:
            ImageUtils.imshow(last_image_input.get_im_gray(), "im_gray")
        if last_image_input.get_im_vis_bgr() is not None:
            ImageUtils.imshow(last_image_input.get_im_vis_bgr(), "Final Keys Overlay")
    return last_keybed_output

def play_y_lag_time_delta_callback(play_y_lag_time_delta_val: float, velocity_consensus: float):
    global play_y_lag_time_delta
    play_y_lag_time_delta = float(play_y_lag_time_delta_val)

def get_actual_happening_time(pts_time: float, force=True) -> float:
    global apply_lag_to_edge
    return Utils.calculate_actual_happening_time(pts_time, apply_lag_to_edge or force, play_y_lag_time_delta, actual_fps)

async def main(keybed_detection_log_level=LogLevel.LOGLEVEL_NONE, note_rain_detection_log_level=LogLevel.LOGLEVEL_NONE):
    pref = Preferences(box_tickness_rate=0.02)
    keybed_runtime_config = RuntimeConfig(AppMode.NOTEBOOK, ProdMode.DEBUG, keybed_detection_log_level)
    note_rain_runtime_config = RuntimeConfig(AppMode.NOTEBOOK, ProdMode.DEBUG, note_rain_detection_log_level)
    processor = Processor(pref, keybed_runtime_config, note_rain_runtime_config, actual_fps=actual_fps, play_y_lag_time_delta_callback_fn=play_y_lag_time_delta_callback)
    midi_tracker = MidiTracker(actual_fps)
    processor.init_keybed_detector_phase()
    
    # The following input sample images are captured/extracted from third-party YouTube videos for
    # limited research/testing use purposes only. See experiment_samples/EXPERIMENT_SOURCES.txt
    # and tests/data/TEST_SOURCES.txt for ownership and limited-use notes.
    
    # UNCOMMENT ONLY ONE OF THE FOLLOWING LINES:

    #test_parent_dir, test_run_name, transpose_octaves = "tests/data/continuity", "test_bittersuite", 0
    test_parent_dir, test_run_name, transpose_octaves = "tests/data/continuity", "test_gulpembe", 0
    #test_parent_dir, test_run_name, transpose_octaves = "tests/data/continuity", "test_rondo_alla_turca", 0

    #test_parent_dir, test_run_name, transpose_octaves = "experiment_samples/continuity", "test_01_one_mans_dream", 0
    #test_parent_dir, test_run_name, transpose_octaves = "experiment_samples/continuity", "test_02_gulpembe", 0
    #test_parent_dir, test_run_name, transpose_octaves = "experiment_samples/continuity", "test_03_gulpembe", 0
    #test_parent_dir, test_run_name, transpose_octaves = "experiment_samples/continuity", "test_04_gulpembe", 0
    #test_parent_dir, test_run_name, transpose_octaves = "experiment_samples/continuity", "test_05_gulpembe", 0
    #test_parent_dir, test_run_name, transpose_octaves = "experiment_samples/continuity", "test_06_bring_me_to_life", 0

    os.makedirs(f"data/debug/continuity/{test_run_name}", exist_ok=True)
    file_list = glob.glob(f"{test_parent_dir}/{test_run_name}" + "/*.png")
    file_list.sort()
    keybed_output = await detect_keybed(processor, file_list)

    if keybed_output is None or keybed_output.evaluation_result is not None:
        raise Exception("keybed_output.keybed_bounds is not valid.")

    state = processor.init_note_rain_pipeline_phase(keybed_output)
    for pts, im_file in enumerate(file_list):
        if pts == 5:
            pass
        pts_time = Utils.pts_to_pts_time(pts, actual_fps)
        try:
            out_pts = int(im_file.split("_")[-1].split(".")[0])
            out_file = f"{pts:04d}_{out_pts:04d}"
        except:
            out_pts = f"{pts:04d}"
        print("Processing: ", out_pts)
        happening_pts, happening_time = get_actual_happening_time(pts_time)
        im_bgr = ImageUtils.read_image(im_file)
        nr_image_input = await ImagePreprocessor.preprocess_for_note_rain(im_bgr, processor.note_rain_pipeline.keybed_output.keybed_bounds, None)
        raw_events, hands_output_ranges = await processor.detect_note_rain(pts, nr_image_input, transpose_octaves=transpose_octaves)
        hands_midi_num_ranges = hands_output_ranges.to_midi_num_ranges()
        note_events = midi_tracker.step_frame(pts, raw_events, hands_midi_num_ranges, 0.0)
        im_vis_bgr = state.get_state(2).image
        note_result_midi_events_str = midi_tracker.report_last_items(pts, pts_time, happening_pts=happening_pts)
        im_vis_bgr, (_, _, _, text_height) = ImageUtils.print_text_on_image(im_vis_bgr, 10, 10, note_result_midi_events_str, font_scale=0.2, fill_color="#082C9A", pad_x=10, pad_y=10)
        active_midi_num_groups = midi_tracker.active_midi_num_groups
        cur_measure_events = midi_tracker.get_active_groups_as_measure_events(hands_midi_num_ranges)
        active_note_placer.set_state(
            cur_measure_events,
            midi_tracker.hands_midi_num_ranges_per_time,
        )
        im_active_notes = await active_note_placer.midi_to_image(
                            None,
                            background_color="#082C9A",
                            foreground_color="white",
                            fixed_size=True,
                            margin_vertical_extra_units=None,
                            output_width=im_vis_bgr.shape[0]//3,
                            output_height=im_vis_bgr.shape[0]//3,
                        )
        print(f"file {pts}/{len(file_list)}: {im_file} -- pts_time: {pts_time} -- note_rain_output:\n" + \
              ("\n".join(map(str, note_events)) if len(note_events) else "[No MIDI events]") + \
              "\nActive MIDI nums:\n" + str(active_midi_num_groups)
        )
        ImageUtils.imshow(im_vis_bgr, caption=f"Processed Frame {out_pts}", filename=f"data/debug/continuity/{test_run_name}/{out_file}.png", save_only=(note_rain_detection_log_level > LogLevel.LOGLEVEL_INFO))

await main(keybed_detection_log_level=LogLevel.LOGLEVEL_NONE, note_rain_detection_log_level=LogLevel.LOGLEVEL_INFO)
